In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import dabench as dab
import numpy as np
import jax
from timeit import default_timer as timer
import pandas as pd
import matplotlib.pyplot as plt
import pickle

In [3]:
%%bash
mkdir -p out_rev/l96

# Define parameters

In [4]:
system_dim= 36
spinup_size = 14400
valid_size = 5000
test_size = 5000
nr_steps = spinup_size + valid_size + test_size
delta_t=0.01
obs_sd = 0.5
sigma_bg = 0.33
sigma_obs = 0.625
analysis_window = 0.1
obs_location_count = 18
random_seed = 5

# Function definition: Backprop 4DVar

We'll need to prep and run Backprop-4DVar many times, so this wraps it all into one function

In [ ]:
def run_etkf(system_dim, nr_steps, spinup_size, valid_size, test_size, 
                       test_run, delta_t, obs_location_count, obs_sd, sigma_bg, 
                       sigma_obs, analysis_window, 
                       random_seed, ensemble_size):
    np_rng = np.random.default_rng(random_seed)
    jax.clear_caches()

    ### Nature Run
    nature_run = dab.data.Lorenz96(system_dim=system_dim, delta_t=delta_t,
                                   store_as_jax=True, random_seed=random_seed)

    x0_initial = np_rng.normal(size=system_dim, scale=1)
    nature_run.generate(n_steps=nr_steps, x0 = x0_initial) 
    # Offset by test size, so we're taking last 5000 of spinup and then the 5000 of valid
    nr_spinup, nr_valid, nr_test = nature_run.split_train_valid_test(
        spinup_size-test_size, valid_size + test_size, 0)

    if not test_run:
        nr_eval = nr_valid
    else:
        nr_eval = nr_test


    ### Observations
    obs_l96 = dab.observer.Observer(
        nr_eval,
        time_indices = np.arange(0, nr_eval.time_dim, 5),
        random_location_count = obs_location_count,
        error_bias = 0.0,
        error_sd = obs_sd,
        random_seed=random_seed,
        stationary_observers=True,
        store_as_jax=True
    )
    obs_vec_l96 = obs_l96.observe()

    
    ### Forecast Model
    model_l96 = dab.data.Lorenz96(system_dim=system_dim, delta_t=delta_t, 
                                  store_as_jax=True, random_seed=random_seed)

    class L96Model(dab.model.Model):                                                                       
        """Defines model wrapper for Lorenz96 to test forecasting."""
        def forecast(self, state_vec, n_steps):
            self.model_obj.generate(x0=state_vec.values, n_steps=n_steps)
            new_vals = self.model_obj.values 

            new_vec = dab.vector.StateVector(values=new_vals, store_as_jax=True)

            return new_vec

    fc_model = L96Model(model_obj=model_l96)
    
    ### Set up DA matrices: H (observation), R (obs error), B (background error)
    obs_times_per_window=2
    total_obs_count = obs_times_per_window*obs_location_count
    H = np.zeros((total_obs_count, system_dim))
    H[np.arange(H.shape[0]), np.tile(obs_vec_l96.location_indices[0],obs_times_per_window)] = 1
    R = (sigma_obs**2)* np.identity(total_obs_count)
    B = (sigma_bg**2)*np.identity(system_dim)

    
    ### Run data assimilation
    da_time_start = timer()
    
    # Prep DA object
    dc = dab.dacycler.ETKF(
        system_dim=system_dim,
        delta_t=nr_eval.delta_t,
        B=B,
        H=H,
        R=R,
        model_obj=fc_model,
        ensemble_dim=ensemble_size,
        multiplicative_inflation=1.01,
        )

    # Generate initial conditions
    cur_tstep = 0
    x0_original = nr_eval.values[cur_tstep] + np_rng.normal(size=(ensemble_size, system_dim,), 
                                                            scale=sigma_bg)
    x0_sv = dab.vector.StateVector(
        values=x0_original,
        store_as_jax=True)
    
    # Execute
    out_statevec = dc.cycle(
        input_state = x0_sv,
        start_time = nr_eval.times[cur_tstep],
        obs_vector = obs_vec_l96,
        analysis_window=analysis_window,
        n_cycles=1000,
        return_forecast=True)
    
    da_time = timer()-da_time_start
    rmse = 0 # Not needed for this 
    
    return out_statevec, rmse, obs_vec_l96, nr_eval, da_time

# For system dim experiments

In [ ]:
ensemble_size=200
out_dict_list_etkf = []

system_dim_list = [6, 20, 36, 72, 144, 256]

for system_dim in system_dim_list:
    
        obs_location_count = int(system_dim/2)
        random_seed = system_dim
        
        run_dict = dict(system_dim=system_dim, 
            nr_steps=nr_steps,
            spinup_size=spinup_size,
            valid_size=valid_size,
            test_size=test_size,
            test_run=False,
            delta_t=delta_t,
            obs_location_count=obs_location_count,
            obs_sd=obs_sd,
            sigma_bg=sigma_bg,
            sigma_obs=sigma_obs,
            analysis_window=analysis_window,
            random_seed=random_seed,
            ensemble_size=ensemble_size)
        
        out_etkf, error_etkf, obs_vec_l96, nr_eval, da_time = run_etkf(**run_dict)
        run_dict['time'] = da_time
        run_dict['rmse'] = error_etkf
        run_dict['run_num'] = 0
        out_file = './out_rev/l96/l96_enkf_statevec_{}dim_v1.pkl'.format(system_dim)
        with open(out_file, 'wb') as f: 
            pickle.dump(out_etkf, f) 
        f.close()

        
        print('Run {}, Time = {}'.format(0,run_dict['time']))
        out_dict_list_etkf.append(run_dict)

In [7]:

fig, axes = plt.subplots(6, 1, sharex = True, figsize = (10, 8))
for j, ax in enumerate(axes):
    ax.plot(out_etkf.times, nr_eval.values[:out_etkf.values.shape[0],j], lw = 3, label = 'True')
    ax.errorbar(out_etkf.times, np.mean(out_etkf.values, axis=1)[:,j],
                yerr=np.ptp(out_etkf.values, axis=1)[:,j], elinewidth=0.5, ecolor='red')
    # ax.plot(obs_vec_l96.times[700:900], obs_vec_l96.values[700:900, np.where(j == obs_vec_l96.location_indices[0])[0]])
    ax.set_ylabel(r'$x_{:d}$'.format(j), fontsize = 16)
ax.set_xlabel('Time (s)')
plt.show()

In [8]:


fig, axes = plt.subplots(6, 1, sharex = True, figsize = (10, 8))
for j, ax in enumerate(axes):
    ax.plot(out_etkf.times, nr_eval.values[:out_etkf.values.shape[0],j], lw = 3, label = 'True')
    ax.errorbar(out_etkf.times, np.mean(out_etkf.values, axis=1)[:,j],
                yerr=np.ptp(out_etkf.values, axis=1)[:,j], elinewidth=0.5, ecolor='red')
    ax.plot(obs_vec_l96.times, obs_vec_l96.values[:, np.where(j == obs_vec_l96.location_indices[0])[0]])
    ax.set_ylabel(r'$x_{:d}$'.format(j), fontsize = 16)
ax.set_xlabel('Time (s)')
# ax.set_xlim(146,148)
plt.show()

# For Heatmap

In [6]:

ensemble_size=100
out_dict_list_etkf_hm = []

system_dim=36
num_obs_list = [6, 12, 18, 24, 30, 36]
obs_error_list = [0.1, 0.2, 0.3, 0.4, 0.5, 0.75, 1.0, 1.5, 2.0]

for obs_location_count in num_obs_list:
    for obs_sd in obs_error_list:

        random_seed = 99 + obs_location_count
        
        run_dict = dict(system_dim=system_dim, 
            nr_steps=nr_steps,
            spinup_size=spinup_size,
            valid_size=valid_size,
            test_size=test_size,
            test_run=False,
            delta_t=delta_t,
            obs_location_count=obs_location_count,
            obs_sd=obs_sd,
            sigma_bg=sigma_bg,
            sigma_obs=sigma_obs,
            analysis_window=analysis_window,
            random_seed=random_seed,
            ensemble_size=ensemble_size)
        
        out_etkf, error_etkf, obs_vec_l96, nr_eval, da_time = run_etkf(**run_dict)
        run_dict['time'] = da_time
        run_dict['rmse'] = error_etkf
        run_dict['run_num'] = 0
        out_file = './out_rev/l96/l96_enkf_statevec_{}obs_{}error_v1.pkl'.format(obs_location_count, obs_sd)
        with open(out_file, 'wb') as f: 
            pickle.dump(out_etkf, f) 
        f.close()

        
        print('Run {}, Time = {}'.format(0,run_dict['time']))
        out_dict_list_etkf_hm.append(run_dict)